## Corregir columna Hora

In [1]:
import pandas as pd
import re

df = pd.read_csv('madrid_accidentes_limpio.csv', low_memory=False)

def limpiar_hora_exacta(texto):
    # Buscamos solo los dígitos de la hora (antes de los dos puntos)
    match = re.search(r'(\d{1,2}):', str(texto))
    if match:
        h = match.group(1) # Extrae solo la hora (ej: "9" o "09")
        # Asegurar que tenga dos dígitos (ej: "09")
        if len(h) == 1: 
            h = '0' + h
        return f"{h}:00:00" # Forzamos minutos y segundos a 00
    
    return "08:00:00" # Valor por defecto si no hay coincidencia

# Aplicar la función a la columna
df['hora'] = df['hora'].apply(limpiar_hora_exacta)
df

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,num_victimas,tipo_accidente,tipo_vehiculo,...,rango_edad,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,id_tipo_accidente,direccion_unica
0,2016-01-01,00:00:00,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,4.0,atropello,turismo,...,de 30 a 34 anos,13.0,despejado,NaN,NaN,NaN,NaN,NaN,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...
1,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,turismo,...,de 30 a 34 anos,15.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
2,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,motocicleta,...,de 25 a 29 años,15.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
3,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
4,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236197,2023-12-31,21:00:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 45 a 49 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
236198,2023-12-31,21:00:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 60 a 64 años,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
236199,2023-12-29,09:00:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,motocicleta hasta 125cc,...,de 45 a 49 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"
236200,2023-12-29,09:00:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,turismo,...,de 21 a 24 años,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0"


## Unir con Tráfico

In [10]:
# Creamos un pequeño "molde" con las filas únicas de fecha y hora
filtro_momentos = df[['fecha', 'hora']].drop_duplicates()

# Convertimos esto a una lista de tuplas para que la búsqueda sea ultra rápida
momentos_exactos = set(zip(filtro_momentos['fecha'], filtro_momentos['hora']))

import pandas as pd

chunks_utiles = []
path_trafico = 'sensores_trafico_madrid_acumulado.csv'

# Asegúrate de que momentos_exactos esté definido antes
# momentos_exactos = set(zip(df_accidentes['fecha'], df_accidentes['hora']))

for chunk in pd.read_csv(path_trafico, sep=';', chunksize=100000):
    # 1. Convertir la columna fecha usando el formato ISO8601
    # Esto lee "2016-01-13 00:00:00" correctamente
    tiempos = pd.to_datetime(chunk['fecha'], format='ISO8601')
    
    # 2. Extraer la fecha limpia y la hora (0-23) de esa misma columna
    chunk['fecha_dt'] = tiempos.dt.date
    chunk['hora_dt'] = tiempos.dt.hour
    
    # 3. Filtrar usando las columnas nuevas que acabamos de crear
    # Comparamos contra el set de (fecha, hora) de accidentes
    mask = chunk.apply(lambda row: (row['fecha_dt'], row['hora_dt']) in momentos_exactos, axis=1)
    
    # Guardamos el trozo filtrado
    chunk_filtrado = chunk[mask].copy()
    
    # Renombramos para que el merge final sea más fácil
    chunk_filtrado = chunk_filtrado.drop(columns=['fecha', 'hora'], errors='ignore')
    chunk_filtrado = chunk_filtrado.rename(columns={'fecha_dt': 'fecha', 'hora_dt': 'hora'})
    
    chunks_utiles.append(chunk_filtrado)

# Unir todo lo encontrado
df_trafico_reducido = pd.concat(chunks_utiles)

print(f"Filtrado completado. Se han encontrado {len(df_trafico_reducido)} filas coincidentes.")

KeyboardInterrupt: 

## Festividades

In [3]:
import pandas as pd

# 1. Cargar los archivos con el encoding correcto para evitar el UnicodeDecodeError
# 'latin-1' permite leer eñes y tildes correctamente en estos archivos
df_locales = pd.read_csv('festivos_locales_historicos.csv', sep=';', encoding='latin-1')
df_regionales = pd.read_csv('festivos_regionales_historicos.csv', sep=';', encoding='latin-1')

# 2. ARREGLAR COLUMNAS DE FECHA
# Convertimos a datetime. format='mixed' ayuda si hay formatos distintos en el mismo archivo
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df_locales['fecha_festivo'] = pd.to_datetime(df_locales['fecha_festivo'], errors='coerce')
df_regionales['fecha_festivo'] = pd.to_datetime(df_regionales['fecha_festivo'], errors='coerce')

# 3. Filtrar festivos (Años 2016-2024 y Municipio Madrid)
df_loc_filt = df_locales[(df_locales['año'].between(2016, 2024)) & 
                         (df_locales['municipio_nombre'] == 'Madrid')]

df_reg_filt = df_regionales[df_regionales['año'].between(2016, 2024)]

# 4. Crear el conjunto (set) de días festivos (usamos .dt.date para comparar solo día/mes/año)
set_festivos = set(df_loc_filt['fecha_festivo'].dt.date) | set(df_reg_filt['fecha_festivo'].dt.date)

# 5. Crear la columna binaria 'es_festivo'
# Eliminamos posibles valores nulos en fecha antes de comparar para evitar errores
df['es_festivo'] = df['fecha'].dt.date.isin(set_festivos).astype(int)

# 6. Guardar el resultado final
df.to_csv('madrid_accidentes_limpio_festivos.csv', index=False)

print("¡Proceso completado con éxito!")
print(f"Total de accidentes en días festivos: {df['es_festivo'].sum()}")

¡Proceso completado con éxito!
Total de accidentes en días festivos: 5902


In [4]:
df

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,num_victimas,tipo_accidente,tipo_vehiculo,...,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,id_tipo_accidente,direccion_unica,es_festivo
0,2016-01-01,00:00:00,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,4.0,atropello,turismo,...,13.0,despejado,NaN,NaN,NaN,NaN,NaN,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,1
1,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,turismo,...,15.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7,1
2,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,motocicleta,...,15.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7,1
3,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300,1
4,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236197,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10,0
236198,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,17.0,despejado,14.0,441152.627,4466350.125,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10,0
236199,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,motocicleta hasta 125cc,...,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0",0
236200,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,turismo,...,1.0,despejado,NaN,439594.878,4473163.747,N,NaN,1.0,"PTA. TOLEDO, 0 0",0


## Coordenadas

In [14]:
import pandas as pd
import numpy as np

# 1. Cargar los archivos
df = pd.read_csv('madrid_accidentes_limpio.csv', low_memory=False)
df_cal = pd.read_csv('callejero_madrid.csv', sep=';', encoding='latin-1', low_memory=False)

# 2. Preparar el Callejero (Estandarización)
# Convertimos UTM a flotante (reemplazando coma por punto)
df_cal['UTMX_ETRS'] = df_cal['UTMX_ETRS'].str.replace(',', '.').astype(float)
df_cal['UTMY_ETRS'] = df_cal['UTMY_ETRS'].str.replace(',', '.').astype(float)

# Función para construir el nombre de la calle
def construir_calle(row):
    # Unimos Clase (CALLE), Partícula (DE) y Nombre (ALCALA)
    partes = [str(row['VIA_CLASE']), str(row['VIA_PAR']), str(row['VIA_NOMBRE'])]
    # Limpiamos nulos y espacios
    res = ' '.join([p.strip() for p in partes if p and str(p).lower() != 'nan']).upper()
    return ' '.join(res.split()) # Quita espacios dobles

df_cal['CALLE_LIMPIA'] = df_cal.apply(construir_calle, axis=1)
df_cal['DIRECCION_FULL'] = df_cal['CALLE_LIMPIA'] + " " + df_cal['NÚMERO'].astype(str)

# 3. Crear diccionarios de búsqueda (para que sea instantáneo)
# Mapa de Dirección exacta (Calle + Número) -> Coordenadas
mapa_exacto_x = df_cal.groupby('DIRECCION_FULL')['UTMX_ETRS'].first().to_dict()
mapa_exacto_y = df_cal.groupby('DIRECCION_FULL')['UTMY_ETRS'].first().to_dict()

# Mapa de Calle (Media de la calle) -> Coordenadas (Para cruces o números inexistentes)
mapa_calle_x = df_cal.groupby('CALLE_LIMPIA')['UTMX_ETRS'].mean().to_dict()
mapa_calle_y = df_cal.groupby('CALLE_LIMPIA')['UTMY_ETRS'].mean().to_dict()

# 4. Limpiar la columna del accidente para el cruce
# Quitamos " NUM " y " KM. " para que coincida con el formato del callejero
def normalizar_accidente(dir_acc):
    if pd.isna(dir_acc): return ""
    res = str(dir_acc).replace(' NUM ', ' ').replace(' KM. ', ' ').upper()
    return ' '.join(res.split())

df['dir_norm'] = df['direccion_unica'].apply(normalizar_accidente)

# 5. IMPUTACIÓN DE COORDENADAS
# Paso A: Intentar coincidencia exacta (Calle + Número)
nulos_antes = df['coordenada_x_utm'].isna().sum()

df['coordenada_x_utm'] = df['coordenada_x_utm'].fillna(df['dir_norm'].map(mapa_exacto_x))
df['coordenada_y_utm'] = df['coordenada_y_utm'].fillna(df['dir_norm'].map(mapa_exacto_y))

# Paso B: Para los que siguen nulos (como cruces), extraer la calle principal y usar su media
def extraer_primera_calle(dir_norm):
    # El separador de cruces suele ser " / " o " - "
    for sep in [' / ', ' - ']:
        if sep in dir_norm:
            return dir_norm.split(sep)[0].strip()
    # Si tiene número al final, intentamos quitarlo para buscar solo la calle
    palabras = dir_norm.split()
    if palabras and palabras[-1].isdigit():
        return ' '.join(palabras[:-1])
    return dir_norm

df['calle_base'] = df['dir_norm'].apply(extraer_primera_calle)

df['coordenada_x_utm'] = df['coordenada_x_utm'].fillna(df['calle_base'].map(mapa_calle_x))
df['coordenada_y_utm'] = df['coordenada_y_utm'].fillna(df['calle_base'].map(mapa_calle_y))

# 6. Limpieza final y guardado
nulos_despues = df['coordenada_x_utm'].isna().sum()
df.drop(columns=['dir_norm', 'calle_base'], inplace=True)
df.to_csv('madrid_accidentes_coordenadas_completas.csv', index=False)

print(f"Proceso finalizado.")
print(f"Nulos recuperados: {nulos_antes - nulos_despues}")
print(f"Nulos restantes: {nulos_despues}")

Proceso finalizado.
Nulos recuperados: 48199
Nulos restantes: 8768


In [15]:
df

,fecha,hora,dia_semana,distrito,localizacion,numero,num_expediente,num_victimas,tipo_accidente,tipo_vehiculo,...,rango_edad,cod_distrito,estado_meteorológico,cod_lesividad,coordenada_x_utm,coordenada_y_utm,positiva_alcohol,positiva_droga,id_tipo_accidente,direccion_unica
0,2016-01-01,00:00:00,VIERNES,puente de vallecas,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...,0,2016/4,4.0,atropello,turismo,...,de 30 a 34 anos,13.0,despejado,NaN,442270.390461,4.446289e+06,NaN,NaN,0.0,AVENIDA DE PABLO NERUDA - CALLE DEL GOLFO DE D...
1,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,turismo,...,de 30 a 34 anos,15.0,despejado,NaN,444388.100000,4.474950e+06,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
2,2016-01-01,01:00:00,VIERNES,ciudad lineal,AVENIDA DEL MARQUES DE CORBERA NUM ...,7,2016/437,1.0,colisión doble,motocicleta,...,de 25 a 29 años,15.0,despejado,NaN,444388.100000,4.474950e+06,NaN,NaN,1.0,AVENIDA DEL MARQUES DE CORBERA NUM 7
3,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
4,2016-01-01,02:00:00,VIERNES,chamartín,AUTOVIA M-30 CALZADA 1 KM. ...,1300,2016/31,1.0,colisión múltiple,turismo,...,de 50 a 54 años,5.0,despejado,NaN,NaN,NaN,NaN,NaN,1.0,AUTOVIA M-30 CALZADA 1 KM. 1300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236197,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 45 a 49 años,17.0,despejado,14.0,441152.627000,4.466350e+06,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
236198,2023-12-31,21:15:00,DOMINGO,villaverde,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA,10,2023S040267,NaN,colisión fronto-lateral,turismo,...,de 60 a 64 años,17.0,despejado,14.0,441152.627000,4.466350e+06,N,NaN,1.0,AVDA. GRAN VIA DE VILLAVERDE / AVDA. ANDALUCIA 10
236199,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,motocicleta hasta 125cc,...,de 45 a 49 años,1.0,despejado,NaN,439594.878000,4.473164e+06,N,NaN,1.0,"PTA. TOLEDO, 0 0"
236200,2023-12-29,09:35:00,VIERNES,centro,"PTA. TOLEDO, 0",0,2023S040277,NaN,alcance,turismo,...,de 21 a 24 años,1.0,despejado,NaN,439594.878000,4.473164e+06,N,NaN,1.0,"PTA. TOLEDO, 0 0"
